# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")

In [4]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [5]:
# Signal 1

signal1 = con.execute(f"""
    SELECT 
        CASE WHEN gsc_avg_position <= 3 THEN '1-3'
             WHEN gsc_avg_position <= 6 THEN '4-6'
             WHEN gsc_avg_position <= 10 THEN '7-10'
             ELSE '11+' END as position_bucket,
        COUNT(*) as n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

signal1

,position_bucket,n,avg_ctr
0,1-3,727362,0.004756
1,11+,1427577,0.001828
2,4-6,787643,0.003946
3,7-10,668479,0.002914


In [6]:
# Signal 2

signal2 = con.execute(f"""
    SELECT 
        CASE WHEN gsc_impressions < 100 THEN 'low'
             WHEN gsc_impressions < 1000 THEN 'medium'
             ELSE 'high' END as impression_bucket,
        COUNT(*) as n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY impression_bucket
    ORDER BY impression_bucket
""").df()

signal2

,impression_bucket,n,avg_ctr
0,high,32419,0.002714
1,low,2972453,0.003086
2,medium,606189,0.003072


**Signal 1 — Position vs CTR (bucket table above):** CONFIRMED. CTR drops steadily as 
position gets worse (0.48% at position 1-3, down to 0.18% at position 11+), with large 
sample sizes in every bucket. This confirms the core assumption behind CTR-fix logic: 
better position should mean better CTR, and it does here.

**Signal 2 — Impression volume vs CTR (bucket table above):** MIXED. CTR does not 
clearly rise or fall with volume — low and medium volume pages actually show slightly 
higher average CTR than high-volume pages. This means volume alone is not a reliable 
CTR signal, so I will not use it to guess whether CTR is "trustworthy." I will still 
use a minimum impression count as a simple filter, but only to exclude pages too small 
to matter, not as a CTR-behavior signal.

**My rule (plain words):** A page is worth flagging if it ranks reasonably well 
(top 10) but its CTR is clearly below what similar-position pages typically get, 
and it gets enough impressions to be worth fixing.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import numpy as np
import pandas as pd

In [8]:
df = con.execute(f"""
    SELECT 
        content_hash_id, client_hash_id,
        gsc_avg_position, gsc_impressions, gsc_clicks,
        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) as actual_ctr
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE 
      AND gsc_impressions > 0
      AND gsc_avg_position <= 10
""").df()

df.shape

(2183484, 6)

In [9]:
df["position_bucket"] = pd.cut(df["gsc_avg_position"], bins=[0, 3, 6, 10])
df["expected_ctr"] = df.groupby("position_bucket")["actual_ctr"].transform("mean")

In [10]:
# minimum impressions filter 
enough_volume = df["gsc_impressions"] >= 100

# the gap between what's expected and what's actually happening
ctr_gap = df["expected_ctr"] - df["actual_ctr"]

df["score"] = np.where(enough_volume, ctr_gap * df["gsc_impressions"], 0)

# reason code: why did it score this way
df["reason_code"] = np.where(
    df["score"] > 0, "LOW_CTR_FOR_POSITION", "NOT_ENOUGH_GAP_OR_VOLUME"
)

# action label: what to actually do
df["action"] = np.where(df["score"] > 0, "REVIEW_TITLE_META", "NO_ACTION")

In [13]:
ranked = df.sort_values("score", ascending=False)

output_dir = os.path.normpath(os.path.join(os.getcwd(), "..", "outputs"))
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "baseline_action_score.csv")

ranked.to_csv(output_path, index=False)
print("Saved to:", output_path)

ranked.head(10)

Saved to: c:\Users\tayyaba\Desktop\Flyrank-ML-Internship\work\outputs\baseline_action_score.csv


,content_hash_id,client_hash_id,gsc_avg_position,gsc_impressions,gsc_clicks,actual_ctr,position_bucket,expected_ctr,score,reason_code,action
2136831,content_44f34c0a90047651,client_23a62021009f63c4,0.083350,40084,1,0.000025,"(0, 3]",0.004918,196.141438,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
27219,content_34a70fea29d15f24,client_62f4a7e64f5e0096,2.764916,39003,2,0.000051,"(0, 3]",0.004918,189.824856,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
1956813,content_fec55986a1868d62,client_73cda7b4e4f265ea,0.181500,33383,0,0.000000,"(0, 3]",0.004918,164.184528,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
1895486,content_44f34c0a90047651,client_23a62021009f63c4,0.132532,32958,0,0.000000,"(0, 3]",0.004918,162.094290,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
1953461,content_44f34c0a90047651,client_23a62021009f63c4,0.142508,32756,2,0.000061,"(0, 3]",0.004918,159.100812,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
2098042,content_fec55986a1868d62,client_73cda7b4e4f265ea,0.083407,31472,0,0.000000,"(0, 3]",0.004918,154.785834,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
1723307,content_44f34c0a90047651,client_23a62021009f63c4,0.117814,30964,1,0.000032,"(0, 3]",0.004918,151.287384,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
1705619,content_44f34c0a90047651,client_23a62021009f63c4,0.088955,30791,2,0.000065,"(0, 3]",0.004918,149.436534,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
2090945,content_44f34c0a90047651,client_23a62021009f63c4,0.238315,30573,2,0.000065,"(0, 3]",0.004918,148.364365,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META
195218,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,0.000311,28973,0,0.000000,"(0, 3]",0.004918,142.495232,LOW_CTR_FOR_POSITION,REVIEW_TITLE_META


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: 40,084 impressions, 
   1 click, position ~0. Confidence: LOW. Wrong if: impressions come from bot/broad 
   traffic, not real users.

2. content_34a70fea29d15f24 — Action: REVIEW_TITLE_META. Reason: 39,003 impressions, 
   2 clicks, position ~2.8. Confidence: MEDIUM. Wrong if: impressions are from 
   irrelevant queries.

3. content_fec55986a1868d62 — Action: REVIEW_TITLE_META. Reason: 33,383 impressions, 
   0 clicks, position ~0.2. Confidence: LOW. Wrong if: page is not meant to be clicked 
   directly (e.g. a redirect or utility page).

4. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: same page as #1, 
   different day, 32,958 impressions, 0 clicks. Confidence: LOW. Wrong if: same issue 
   as #1, and this repeat shows the rule is flagging one page many times.

5. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: same page again, 
   32,756 impressions, 2 clicks. Confidence: LOW. Wrong if: same as #1 — page keeps 
   repeating in the top 10, not a diverse list.

6. content_fec55986a1868d62 — Action: REVIEW_TITLE_META. Reason: same page as #3, 
   31,472 impressions, 0 clicks. Confidence: LOW. Wrong if: same reasoning as #3.

7. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: same page again, 
   30,964 impressions, 1 click. Confidence: LOW. Wrong if: same as #1.

8. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: same page again, 
   30,791 impressions, 2 clicks. Confidence: LOW. Wrong if: same as #1.

9. content_44f34c0a90047651 — Action: REVIEW_TITLE_META. Reason: same page again, 
   30,573 impressions, 2 clicks. Confidence: LOW. Wrong if: same as #1.

10. content_9c057b66c30a3abb — Action: REVIEW_TITLE_META. Reason: 28,973 impressions, 
    0 clicks, position ~0. Confidence: LOW. Wrong if: page isn't meant for direct 
    clicks, or impressions are inflated.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** The biggest problem is that `content_44f34c0a90047651` appears 6 times 
in the top 10 — it is really one page, not six different opportunities. This happens 
because the rule scores each page-day separately instead of combining a page's data 
across days first. A better version of this rule would group by content_id before 
ranking, so the top 10 shows 10 different pages, not the same page repeated.

The `avg_position` values near 0 (0.08, 0.18, 0.2) are also suspicious — a real search 
position is never below 1. Combined with near-zero CTR on tens of thousands of 
impressions, this looks like unusual or bot-like traffic rather than a normal, 
fixable CTR problem. These picks should not be trusted without checking the raw 
query-level data behind them.

**Leakage check:** No future data or label-derived columns were used. The score is 
built only from `gsc_avg_position`, `gsc_impressions`, and `gsc_clicks` — all real, 
already-recorded values for that day, not calculated from the future or from the 
target itself. `expected_ctr` was calculated only from the same day's position 
buckets, not from any later outcome.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.